# Scraping and scaling archive.org videos:

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import subprocess
from pathlib import Path

In [ ]:
def process_video(input_file, output_file):
    """
    Scales, pads, and compresses the video to make it smaller in size and pixels, ready for machine learning.

    Parameters:
        input_file (str): Path to the input video file.
        output_file (str): Path to the output video file.
    """
    print(f"Processing {input_file}...")

    # Set a target resolution (e.g., 224x224, 128x128)
    target_resolution = "224x224"

    # Set a bitrate to reduce the file size (e.g., 500k for smaller files)
    target_bitrate = "500k"

    # Use ffmpeg to scale, pad, and compress the video
    command = [
        "ffmpeg",
        "-i", input_file, 
        "-vf", f"scale={target_resolution}, pad=max(iw\,ih):max(iw\,ih):(ow-iw)/2:(oh-ih)/2",  # Scale and pad
        "-b:v", target_bitrate,  # Set target bitrate for video compression
        "-c:a", "aac",  # Compress audio with AAC codec
        "-preset", "ultrafast",  # Use ultrafast encoding for speed (you can change to "fast" or "medium" for better compression)
        "-y",  # Overwrite output file without asking
        output_file
    ]

    try:
        subprocess.run(command, check=True)
        print(f"Processed video saved to {output_file}")
    except subprocess.CalledProcessError as e:
        print(f"Failed to process {input_file}: {e}")

In [ ]:
def scrape_and_process_videos(archive_url, download_dir="./data/downloads", output_dir="./data/processed_videos"):
    """
    Scrapes the given archive.org URL for MPEG4 files, downloads them, and preprocesses them.

    Parameters:
        archive_url (str): The URL from archive.org.
        download_dir (str): Directory to save the downloaded files.
        output_dir (str): Directory to save processed video files.
    """
    # Ensure download and output directories exist
    os.makedirs(download_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    # Get the page content
    print("Fetching archive.org page...")
    response = requests.get(archive_url)
    if response.status_code != 200:
        print("Failed to fetch the page.")
        return

    soup = BeautifulSoup(response.text, 'html.parser')

    # Find MPEG4 files under download options
    print("Finding MPEG4 files...")
    links = soup.find_all('a', href=True)
    mpeg4_links = [link['href'] for link in links if link['href'].endswith('.mp4')]

    if not mpeg4_links:
        print("No MPEG4 files found.")
        return

    # Download each file
    for link in mpeg4_links:
        file_url = f"https://archive.org{link}"
        file_name = os.path.basename(link)
        file_path = os.path.join(download_dir, file_name)

        print(f"Downloading {file_name}...")
        with requests.get(file_url, stream=True) as r:
            with open(file_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

        print(f"Downloaded {file_name}")

        # Process the video
        process_video(file_path, os.path.join(output_dir, file_name))

In [ ]:
if __name__ == "__main__":
    # Example usage
    archive_url = "https://archive.org/details/bronxny-RELEASE_Film_OPEN"
    scrape_and_process_videos(archive_url)

# Detecting shot size using opencv:

In [ ]:
import cv2

Problem with the following code: It only detects faces from the front.

In [ ]:
def detect_face(frame):
    """
    Detects a face (only from the front) in the given frame and returns its bounding box.
    Parameters:
        frame (numpy.ndarray): The current video frame.
    Returns:
        tuple: Bounding box of the face (x, y, w, h) if a face is detected, None otherwise.
    """
    # Load the pre-trained Haar Cascade face detector
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    # Convert the frame to grayscale (Haar cascade works with grayscale images)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # If at least one face is detected, return the first one
    if len(faces) > 0:
        return faces[0]  # (x, y, w, h)

    # No face detected
    return None

In [ ]:
def classify_shot(frame):
    """
    Classifies the shot type based on the face's size relative to the frame.
    Parameters:
        frame (numpy.ndarray): The current video frame.
    Returns:
        str: Shot type ("close-up", "medium shot", "long shot", or "no face detected").
    """
    face = detect_face(frame)
    
    if face is not None:
        face_height = face[3]  
        frame_height = frame.shape[1]  
        ratio = face_height / frame_height

        if 0.25 <= ratio:
            return "close-up"
        elif 0.1 < ratio < 0.25:
            return "medium shot"
        else:
            return "long shot"
    return "no face detected"

In [ ]:
def detect_shot_size (video_path):
    """
    Classifies the shot size in a video file.
    Parameters:
        video_path (file path): The path to a video file.
    Returns:
        For the time, only prints the shot sizes and returns nothing. 
    """
    cap = cv2.VideoCapture(video_path)
    # Initialize a counter for frames
    frame_index = 0  
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Process every tenth frame
        if frame_index % 10 == 0:
            shot_type = classify_shot(frame)
            print(shot_type)
        
        frame_index += 1
    
    cap.release()

Detecting shot size in example file 1.mp4:

In [ ]:
detect_shot_size ("./data/1.mp4")

Detecting shot size in example file 2.mp4:

In [ ]:
detect_shot_size ("./data/2.mp4")